# 06 · Recoverable rows and failing pipelines

Record exceptions can be quarantined; dataset exceptions can invalidate a batch; infrastructure exceptions can fail the pipeline. Catch an expected error narrowly, record context, and do not convert every exception into a successful empty output.


## Environment
Upload the prepared sample files before the session, then use notebook 00 to check the configured storage. This notebook then runs independently, top to bottom. Spark 3.5 is the target; no Hive catalog is used. Set `BASE_PATH` in the following cell or set `DQ_BASE_PATH` in the driver environment.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("OrderDataQuality").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # tiny teaching datasets only
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
BASE_PATH = os.environ.get(
    "DQ_BASE_PATH", "s3://YOUR-BUCKET/training/order-quality"
).rstrip("/")
# On EMR/Glue edit the default above if driver environment variables are unavailable.
# Spark VM: hdfs:///user/student/order-quality ; local: file:///tmp/order-quality
assert (
    "YOUR-BUCKET" not in BASE_PATH
), "Set DQ_BASE_PATH or edit BASE_PATH before running"
PROCESSING_DATE = os.environ.get("DQ_PROCESSING_DATE", "2024-01-03")
RUN_ID = uuid.uuid4().hex
RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH = [
    f"{BASE_PATH}/{layer}"
    for layer in ["raw", "bronze", "silver", "gold", "quarantine", "audit"]
]
print("Spark", spark.version, "storage", BASE_PATH, "run", RUN_ID)


## Missing input and empty input
A missing path is an I/O exception. An empty but correctly typed DataFrame is a business condition. Spark is lazy: put the action inside the try block.


In [ ]:
from pyspark.errors import AnalysisException

try:
    spark.read.json(f"{RAW_PATH}/missing_{RUN_ID}").show()
except AnalysisException as exc:
    print("Expected missing-input failure:", type(exc).__name__)
empty = spark.createDataFrame([], "order_id string, amount decimal(18,2)")
print(
    "Empty dataset action:",
    "FAIL DATASET" if empty.limit(1).count() == 0 else "CONTINUE",
)


## Explicit policies by failure scope
This table is a policy, not a claim that Spark classifies your business exceptions automatically. A corrupt envelope can be rejected individually; excessive corruption can stop the whole dataset.


In [ ]:
spark.createDataFrame(
    [
        ("missing path", "pipeline", "FAIL PIPELINE"),
        ("empty mandatory input", "dataset", "FAIL DATASET"),
        ("corrupt JSON line", "record", "QUARANTINE"),
        ("missing required column", "dataset", "FAIL DATASET"),
        ("unknown customer", "record", "REJECT RECORD"),
        ("late record allowed by SLA", "record", "WARN"),
        ("reject rate above limit", "dataset", "FAIL DATASET"),
        ("missing shipment date", "record", "QUARANTINE"),
        ("unexplained reconciliation difference", "pipeline", "FAIL PIPELINE"),
        ("output write denied", "pipeline", "FAIL PIPELINE"),
    ],
    "condition string, scope string, action string",
).show(truncate=False)


## Output failure without breaking permissions
Write a run-scoped destination once, then deliberately attempt to create it again. `errorifexists` makes reruns visible. In a real job, log the run ID and re-raise unexpected write failures; do not retry authorization or schema errors blindly.


In [ ]:
destination = f"{AUDIT_PATH}/exception_demo/{RUN_ID}"
spark.range(1).write.mode("errorifexists").parquet(destination)
try:
    spark.range(2).write.mode("errorifexists").parquet(destination)
except AnalysisException as exc:
    print("Expected existing-output failure:", type(exc).__name__)
assert spark.read.parquet(destination).count() == 1


## Exercise
A downstream consumer must never see partial Gold data. Write to a new run prefix, persist audit results on failure too, and expose only a successful run manifest. An S3 directory rename is not an atomic transaction. Automated retries need stable source batch identity and an idempotency policy; these notebooks generate a new run ID for each lesson execution.
